# Stage 01 - Parse ADAN-86 CellML into a Solver-Ready Arterial Network

This notebook parses the public ADAN-86 CellML model and converts it into a clean arterial network representation for a simplified 1D pulse-wave solver.

The goal of this stage is not to simulate blood flow yet.  
The goal is to extract:

- arterial vessel components
- parent-child topology
- vessel geometry
- wall stiffness parameters
- derived physical quantities
- solver-ready arrays
- terminal vessels for Windkessel boundary conditions

The output of this notebook is a compact parsed dataset that will be used by the next notebook to simulate pressure-wave propagation through the arterial tree.

## IMPORT AND SETUP

In [ ]:
from pathlib import Path
import networkx as nx
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from matplotlib.collections import LineCollection
from IPython.display import HTML, display
from scipy.integrate import trapezoid
from collections import defaultdict, deque
import pandas as pd
import numpy as np
import time, re, pickle

cellml_path = Path("main_ADAN-86.cellml")
tree = ET.parse(cellml_path)
root = tree.getroot()

## Load and inspect the CellML model

The ADAN-86 model is stored as CellML, which is an XML-based model format.

Before building a solver,  first inspect the structure of the file:

- what components exist
- which components represent vessels
- how components are connected
- where geometry and stiffness parameters are stored

This exploratory part helps  understand how to convert the original CellML model into a simpler numerical network.

In [2]:
# namespace CellML
ns = {"cellml": "http://www.cellml.org/cellml/1.1#"}

components = []

for comp in root.findall("cellml:component", ns):
    name = comp.attrib.get("name")
    variables = [var.attrib.get("name") for var in comp.findall("cellml:variable", ns)]
    components.append({
        "component": name,
        "n_variables": len(variables),
        "variables": variables
    })

comp_df = pd.DataFrame(components)

print("Number of components:", len(comp_df))
display(comp_df.head(30))

Number of components: 107


,component,n_variables,variables
0,environment,1,[time]
1,Systemic,87,"[t, C_svl, C_svn, C_svc, C_ivl, C_ivn, C_ivc, ..."
2,Pulmonary,29,"[t, C_pas, C_pat, C_par, C_pcp, C_pvn, C_pvc, ..."
3,Heart,73,"[t, mt, mta, T, CQ_trv, CQ_puv, CQ_miv, CQ_aov..."
4,ascending_aorta_C0,8,"[u, v, E, r_p, r_d, h, l, q_0]"
5,aortic_arch_C2,8,"[u, v, E, r_p, r_d, h, l, q_0]"
6,brachiocephalic_trunk_C4,8,"[u, v, E, r_p, r_d, h, l, q_0]"
7,aortic_arch_C46,8,"[u, v, E, r_p, r_d, h, l, q_0]"
8,aortic_arch_C64,8,"[u, v, E, r_p, r_d, h, l, q_0]"
9,aortic_arch_C94,8,"[u, v, E, r_p, r_d, h, l, q_0]"


In [3]:
pd.set_option("display.max_rows", 200)

display(comp_df[["component", "n_variables"]].sort_values("component"))

,component,n_variables
3,Heart,73
2,Pulmonary,29
1,Systemic,87
15,abdominal_aorta_C114,8
16,abdominal_aorta_C136,8
17,abdominal_aorta_C164,8
18,abdominal_aorta_C176,8
19,abdominal_aorta_C188,8
20,abdominal_aorta_C192,8
77,anterior_tibial_T3_L208,8


## Identify arterial vessel components

The arterial vessel components share a common set of variables:

- `u`
- `v`
- `E`
- `r_p`
- `r_d`
- `h`
- `l`
- `q_0`

Use this variable pattern to separate true vessel components from other CellML components such as the heart, systemic circulation, pulmonary circulation, and parameter blocks.

This gives the computational arterial segments that will later become the 1D network.

In [4]:
required_vars = {"u", "v", "E", "r_p", "r_d", "h", "l", "q_0"}

vessel_df = comp_df[comp_df["variables"].apply(lambda xs: required_vars.issubset(set(xs)))].copy()
print(f"Vessel components: {len(vessel_df)}")
display(vessel_df[["component", "n_variables"]].head(20))

Vessel components: 103


,component,n_variables
4,ascending_aorta_C0,8
5,aortic_arch_C2,8
6,brachiocephalic_trunk_C4,8
7,aortic_arch_C46,8
8,aortic_arch_C64,8
9,aortic_arch_C94,8
10,thoracic_aorta_C96,8
11,thoracic_aorta_C100,8
12,thoracic_aorta_C104,8
13,thoracic_aorta_C108,8


### Observation

The parser finds 103 computational vessel components.

This is expected.  
Although the source model is commonly referred to as ADAN-86, some anatomical arteries are split into multiple computational segments in the CellML representation.


In [5]:
def strip_numeric_suffix(name):
    return re.sub(r"_(C|R|L)\d+$", "", name)

vessel_df["base_name"] = vessel_df["component"].apply(strip_numeric_suffix)

display(vessel_df.groupby("base_name").size().sort_values(ascending=False).head(30))

base_name
superior_mesenteric_T4       7
abdominal_aorta              6
thoracic_aorta               5
aortic_arch                  4
subclavian                   4
popliteal                    4
femoral                      4
ulnar_T2                     4
splenic_T2                   3
external_carotid_T2          2
external_iliac               2
anterior_tibial_T3           2
brachial                     2
common_interosseous          2
common_carotid               2
axillary                     2
profundus_T2                 2
superior_segmental_T4        2
tibiofibular_trunk           2
radial_T1                    2
vertebral                    2
common_iliac                 2
internal_carotid             2
inferior_segmental_T5        2
posterior_interosseous_T3    2
posterior_tibial_T4          2
posterior_intercostal_T2     2
posterior_intercostal_T1     2
internal_iliac_T1            2
renal                        2
dtype: int64

## Inspect CellML connections

CellML connections define how components exchange variables.

For this project, the most important connections are those between arterial vessel modules.  
These connections allow  to reconstruct the parent-child topology of the arterial tree.

The original CellML model contains more than just arteries, including heart and circulation modules.  
For the simplified animation model,  keep the arterial topology and later replace the full closed-loop circulation with:

- a prescribed inlet flow waveform at the root
- 3-element Windkessel models at terminal vessels

In [6]:
connections = []

for conn in root.findall("cellml:connection", ns):
    comps = conn.findall("cellml:map_components", ns)
    if len(comps) == 0:
        continue
    comp1 = comps[0].attrib.get("component_1")
    comp2 = comps[0].attrib.get("component_2")

    var_maps = []

    for vm in conn.findall("cellml:map_variables", ns):
        v1 = vm.attrib.get("variable_1")
        v2 = vm.attrib.get("variable_2")
        var_maps.append((v1, v2))

    connections.append({
        "component_1": comp1,
        "component_2": comp2,
        "variable_maps": var_maps
    })

conn_df = pd.DataFrame(connections)
print("Connections:", len(conn_df))

display(conn_df.head(20))

Connections: 419


,component_1,component_2,variable_maps
0,Heart,Systemic,"[(v_sup_vc, v_svc), (v_inf_vc, v_ivc), (u_ra, ..."
1,Heart,Pulmonary,"[(u_par, u_pas), (v_pvn, v_pvn), (u_la, u_la),..."
2,Parameters_Systemic,Systemic,"[(C_svl, C_svl), (C_svn, C_svn), (C_svc, C_svc..."
3,Parameters_Pulmonary,Pulmonary,"[(C_pas, C_pas), (C_pat, C_pat), (C_par, C_par..."
4,Parameters_Heart,Heart,"[(T, T), (CQ_trv, CQ_trv), (CQ_puv, CQ_puv), (..."
5,environment,Heart,"[(time, t)]"
6,environment,Systemic,"[(time, t)]"
7,environment,Pulmonary,"[(time, t)]"
8,Systemic,posterior_intercostal_T1_R98_module,"[(u_ivl, u_out), (v_posterior_intercostal_T1_R..."
9,Systemic,posterior_intercostal_T1_L102_module,"[(u_ivl, u_out), (v_posterior_intercostal_T1_L..."


In [7]:
display(conn_df[ conn_df["component_1"].str.contains("aorta", case=False, na=False)
        | conn_df["component_2"].str.contains("aorta", case=False, na=False)])

,component_1,component_2,variable_maps
59,thoracic_aorta_C96_module,thoracic_aorta_C100_module,"[(v_out_1, v), (u, u_in)]"
60,thoracic_aorta_C96_module,posterior_intercostal_T1_R98_module,"[(v_out_2, v), (u, u_in)]"
61,thoracic_aorta_C100_module,thoracic_aorta_C104_module,"[(v_out_1, v), (u, u_in)]"
62,thoracic_aorta_C100_module,posterior_intercostal_T1_L102_module,"[(v_out_2, v), (u, u_in)]"
63,thoracic_aorta_C104_module,thoracic_aorta_C108_module,"[(v_out_1, v), (u, u_in)]"
64,thoracic_aorta_C104_module,posterior_intercostal_T2_R106_module,"[(v_out_2, v), (u, u_in)]"
65,thoracic_aorta_C108_module,thoracic_aorta_C112_module,"[(v_out_1, v), (u, u_in)]"
66,thoracic_aorta_C108_module,posterior_intercostal_T2_L110_module,"[(v_out_2, v), (u, u_in)]"
67,abdominal_aorta_C114_module,abdominal_aorta_C136_module,"[(v_out_1, v), (u, u_in)]"
68,abdominal_aorta_C114_module,celiac_trunk_C116_module,"[(v_out_2, v), (u, u_in)]"


## Reconstruct arterial topology

The next step is to extract direct vessel-to-vessel connections.

The topology tells  which vessel is upstream and which vessels are downstream.  
This is essential for the 1D solver because pressure and flow waves must propagate through bifurcations.

The extracted topology will be represented as edges:

```text
parent vessel → child vessel

In [ ]:
def clean_module_name(name):
    """
    Remove the module suffix from a CellML component name.

    Args:
        name: Component or module name.

    Returns:
        Clean vessel name, or None if input is None.
    """
    if name is None:
        return None
    return name.replace("_module", "")

def is_module_edge(row):
    """
    Check whether a connection row represents a vessel-to-vessel module edge.

    Args:
        row: Connection DataFrame row containing component names
             and variable mappings.

    Returns:
        True if the row connects two vessel modules, otherwise False.
    """
    c1 = row["component_1"]
    c2 = row["component_2"]

    if not isinstance(c1, str) or not isinstance(c2, str):
        return False

    if not c1.endswith("_module") or not c2.endswith("_module"):
        return False

    maps = set(row["variable_maps"])

    # vessel outlet -> vessel inlet
    valid_patterns = [
        ("v_out", "v"),
        ("v_out_1", "v"),
        ("v_out_2", "v"),
        ("u", "u_in"),
    ]

    return any(p in maps for p in valid_patterns)

topo_edges = conn_df[conn_df.apply(is_module_edge, axis=1)].copy()

topo_edges["parent"] = topo_edges["component_1"].apply(clean_module_name)
topo_edges["child"] = topo_edges["component_2"].apply(clean_module_name)

topology_df = topo_edges[["parent", "child", "variable_maps"]].copy()

print("Topology edges:", len(topology_df))
display(topology_df.head(30))

Topology edges: 102


,parent,child,variable_maps
51,aortic_arch_C2,brachiocephalic_trunk_C4,"[(v_out_1, v), (u, u_in)]"
52,aortic_arch_C2,aortic_arch_C46,"[(v_out_2, v), (u, u_in)]"
53,brachiocephalic_trunk_C4,common_carotid_R6,"[(v_out_1, v), (u, u_in)]"
54,brachiocephalic_trunk_C4,subclavian_R28,"[(v_out_2, v), (u, u_in)]"
55,aortic_arch_C46,aortic_arch_C64,"[(v_out_1, v), (u, u_in)]"
56,aortic_arch_C46,common_carotid_L48,"[(v_out_2, v), (u, u_in)]"
57,aortic_arch_C64,aortic_arch_C94,"[(v_out_1, v), (u, u_in)]"
58,aortic_arch_C64,subclavian_L66,"[(v_out_2, v), (u, u_in)]"
59,thoracic_aorta_C96,thoracic_aorta_C100,"[(v_out_1, v), (u, u_in)]"
60,thoracic_aorta_C96,posterior_intercostal_T1_R98,"[(v_out_2, v), (u, u_in)]"


In [9]:
all_vessels = set(vessel_df["component"])
parents = set(topology_df["parent"])
children = set(topology_df["child"])

print("Vessels:", len(all_vessels))
print("Parents in topology:", len(parents))
print("Children in topology:", len(children))
print("Nodes in topology:", len(parents | children))

print("\nVessels not in topology:")
display(sorted(all_vessels - (parents | children)))

print("\nTopology nodes not in vessel components:")
display(sorted((parents | children) - all_vessels))

Vessels: 103
Parents in topology: 60
Children in topology: 102
Nodes in topology: 103

Vessels not in topology:


[]


Topology nodes not in vessel components:


[]

In [10]:
# root = vessel that is never a child
roots = sorted(all_vessels - children)

# terminals = vessels that are never parents
terminals = sorted(all_vessels - parents)

print("Roots:", roots)
print("Number of roots:", len(roots))

print("\nNumber of terminals:", len(terminals))
display(terminals[:30])

# children count per parent
children_count = (topology_df.groupby("parent").size().rename("n_children").reset_index())

node_df = pd.DataFrame({"vessel": sorted(all_vessels)})

node_df = node_df.merge(children_count, left_on="vessel",right_on="parent", how="left")

node_df["n_children"] = node_df["n_children"].fillna(0).astype(int)
node_df = node_df.drop(columns=["parent"])

display(node_df["n_children"].value_counts().sort_index())

display(node_df.sort_values(["n_children", "vessel"], ascending=[False, True]).head(30))

Roots: ['ascending_aorta_C0']
Number of roots: 1

Number of terminals: 43


['anterior_tibial_T3_L208',
 'anterior_tibial_T3_R230',
 'dorsal_pancreatic_T1_C124',
 'external_carotid_T2_L62',
 'external_carotid_T2_R26',
 'hepatic_artery_proper_left_branch_C132',
 'hepatic_artery_proper_right_branch_C134',
 'ileal_4_T12_C156',
 'ileal_6_T13_C160',
 'ileocolic_T9_C152',
 'inferior_mesenteric_T5_C190',
 'inferior_segmental_T5_L170',
 'inferior_segmental_T5_R184',
 'internal_carotid_L50',
 'internal_carotid_R8',
 'internal_iliac_T1_L196',
 'internal_iliac_T1_R218',
 'jejunal_3_T10_C144',
 'jejunal_6_T11_C148',
 'left_gastric_T3_C120',
 'middle_colic_T8_C140',
 'posterior_intercostal_T1_L102',
 'posterior_intercostal_T1_R98',
 'posterior_intercostal_T2_L110',
 'posterior_intercostal_T2_R106',
 'posterior_interosseous_T3_L88',
 'posterior_interosseous_T3_R40',
 'posterior_tibial_T4_L214',
 'posterior_tibial_T4_R236',
 'profundus_T2_L202']

n_children
0    43
1    18
2    42
Name: count, dtype: int64

,vessel,n_children
0,abdominal_aorta_C114,2
1,abdominal_aorta_C136,2
2,abdominal_aorta_C164,2
3,abdominal_aorta_C176,2
4,abdominal_aorta_C188,2
5,abdominal_aorta_C192,2
8,aortic_arch_C2,2
9,aortic_arch_C46,2
10,aortic_arch_C64,2
15,brachial_L82,2


In [11]:
children_map = defaultdict(list)

for _, row in topology_df.iterrows():
    children_map[row["parent"]].append(row["child"])

children_map = dict(children_map)

# sanity preview
for parent in list(children_map.keys())[:10]:
    print(parent, "->", children_map[parent])

aortic_arch_C2 -> ['brachiocephalic_trunk_C4', 'aortic_arch_C46']
brachiocephalic_trunk_C4 -> ['common_carotid_R6', 'subclavian_R28']
aortic_arch_C46 -> ['aortic_arch_C64', 'common_carotid_L48']
aortic_arch_C64 -> ['aortic_arch_C94', 'subclavian_L66']
thoracic_aorta_C96 -> ['thoracic_aorta_C100', 'posterior_intercostal_T1_R98']
thoracic_aorta_C100 -> ['thoracic_aorta_C104', 'posterior_intercostal_T1_L102']
thoracic_aorta_C104 -> ['thoracic_aorta_C108', 'posterior_intercostal_T2_R106']
thoracic_aorta_C108 -> ['thoracic_aorta_C112', 'posterior_intercostal_T2_L110']
abdominal_aorta_C114 -> ['abdominal_aorta_C136', 'celiac_trunk_C116']
abdominal_aorta_C136 -> ['abdominal_aorta_C164', 'superior_mesenteric_T4_C138']


In [12]:
root = roots[0]

order = []
queue = deque([root])

while queue:
    node = queue.popleft()
    order.append(node)

    for child in children_map.get(node, []):
        queue.append(child)

print("Traversal order length:", len(order))
print("First 20:")
print(order[:20])

print("\nLast 20:")
print(order[-20:])

Traversal order length: 103
First 20:
['ascending_aorta_C0', 'aortic_arch_C2', 'brachiocephalic_trunk_C4', 'aortic_arch_C46', 'common_carotid_R6', 'subclavian_R28', 'aortic_arch_C64', 'common_carotid_L48', 'internal_carotid_R8', 'external_carotid_T2_R26', 'subclavian_R30', 'vertebral_R272', 'aortic_arch_C94', 'subclavian_L66', 'internal_carotid_L50', 'external_carotid_T2_L62', 'axillary_R32', 'thoracic_aorta_C96', 'subclavian_L78', 'vertebral_L2']

Last 20:
['ileal_4_T12_C156', 'superior_mesenteric_T4_C158', 'femoral_R222', 'femoral_L200', 'ileal_6_T13_C160', 'superior_mesenteric_T4_C162', 'profundus_T2_R224', 'femoral_R226', 'profundus_T2_L202', 'femoral_L204', 'popliteal_R228', 'popliteal_L206', 'anterior_tibial_T3_R230', 'popliteal_R232', 'anterior_tibial_T3_L208', 'popliteal_L210', 'tibiofibular_trunk_R234', 'tibiofibular_trunk_L212', 'posterior_tibial_T4_R236', 'posterior_tibial_T4_L214']


In [13]:
assert len(order) == len(all_vessels)
assert set(order) == all_vessels

print("✓ Graph traversal covers all vessels")

✓ Graph traversal covers all vessels


In [14]:
parent_map = {}

for _, row in topology_df.iterrows():
    parent_map[row["child"]] = row["parent"]

node_df = pd.DataFrame({"vessel": order})

node_df["order_index"] = range(len(node_df))
node_df["parent"] = node_df["vessel"].map(parent_map)
node_df["children"] = node_df["vessel"].map(lambda x: children_map.get(x, []))
node_df["n_children"] = node_df["children"].apply(len)
node_df["is_root"] = node_df["parent"].isna()
node_df["is_terminal"] = node_df["n_children"].eq(0)

display(node_df.head(30))

print("nodes:", len(node_df))
print("roots:", node_df["is_root"].sum())
print("terminals:", node_df["is_terminal"].sum())
print("bifurcations:", (node_df["n_children"] == 2).sum())
print("linear:", (node_df["n_children"] == 1).sum())

,vessel,order_index,parent,children,n_children,is_root,is_terminal
0,ascending_aorta_C0,0,NaN,[aortic_arch_C2],1,True,False
1,aortic_arch_C2,1,ascending_aorta_C0,"[brachiocephalic_trunk_C4, aortic_arch_C46]",2,False,False
2,brachiocephalic_trunk_C4,2,aortic_arch_C2,"[common_carotid_R6, subclavian_R28]",2,False,False
3,aortic_arch_C46,3,aortic_arch_C2,"[aortic_arch_C64, common_carotid_L48]",2,False,False
4,common_carotid_R6,4,brachiocephalic_trunk_C4,"[internal_carotid_R8, external_carotid_T2_R26]",2,False,False
5,subclavian_R28,5,brachiocephalic_trunk_C4,"[subclavian_R30, vertebral_R272]",2,False,False
6,aortic_arch_C64,6,aortic_arch_C46,"[aortic_arch_C94, subclavian_L66]",2,False,False
7,common_carotid_L48,7,aortic_arch_C46,"[internal_carotid_L50, external_carotid_T2_L62]",2,False,False
8,internal_carotid_R8,8,common_carotid_R6,[],0,False,True
9,external_carotid_T2_R26,9,common_carotid_R6,[],0,False,True


nodes: 103
roots: 1
terminals: 43
bifurcations: 42
linear: 18


## Extract vessel parameters

After reconstructing topology, extract physical vessel parameters from the CellML parameter file.

For each vessel:

- length
- proximal radius
- distal radius
- Young's modulus
- wall thickness or wall-thickness scaling parameter

These values define the geometry and mechanical properties of each compliant tube.

They are later used to compute reference area, stiffness, wave speed, and impedance.

In [15]:
param_edges = conn_df[
    (conn_df["component_1"] == "Parameters_Systemic")
    &
    (conn_df["component_2"].str.endswith("_module", na=False))
].copy()

param_edges["vessel"] = param_edges["component_2"].str.replace("_module", "", regex=False)

print("Parameter edges:", len(param_edges))
display(param_edges.head(10))

Parameter edges: 103


,component_1,component_2,variable_maps,vessel
256,Parameters_Systemic,ascending_aorta_C0_module,"[(l_ascending_aorta_C0, l), (E_ascending_aorta...",ascending_aorta_C0
257,Parameters_Systemic,aortic_arch_C2_module,"[(l_aortic_arch_C2, l), (E_aortic_arch_C2, E),...",aortic_arch_C2
258,Parameters_Systemic,brachiocephalic_trunk_C4_module,"[(l_brachiocephalic_trunk_C4, l), (E_brachioce...",brachiocephalic_trunk_C4
259,Parameters_Systemic,aortic_arch_C46_module,"[(l_aortic_arch_C46, l), (E_aortic_arch_C46, E...",aortic_arch_C46
260,Parameters_Systemic,aortic_arch_C64_module,"[(l_aortic_arch_C64, l), (E_aortic_arch_C64, E...",aortic_arch_C64
261,Parameters_Systemic,aortic_arch_C94_module,"[(l_aortic_arch_C94, l), (E_aortic_arch_C94, E...",aortic_arch_C94
262,Parameters_Systemic,thoracic_aorta_C96_module,"[(l_thoracic_aorta_C96, l), (E_thoracic_aorta_...",thoracic_aorta_C96
263,Parameters_Systemic,thoracic_aorta_C100_module,"[(l_thoracic_aorta_C100, l), (E_thoracic_aorta...",thoracic_aorta_C100
264,Parameters_Systemic,thoracic_aorta_C104_module,"[(l_thoracic_aorta_C104, l), (E_thoracic_aorta...",thoracic_aorta_C104
265,Parameters_Systemic,thoracic_aorta_C108_module,"[(l_thoracic_aorta_C108, l), (E_thoracic_aorta...",thoracic_aorta_C108


In [16]:
first = param_edges.iloc[0]
print(first["component_2"])
print(first["variable_maps"])

ascending_aorta_C0_module
[('l_ascending_aorta_C0', 'l'), ('E_ascending_aorta_C0', 'E'), ('rp_ascending_aorta_C0', 'r_p'), ('rd_ascending_aorta_C0', 'r_d'), ('z_ascending_aorta_C0', 'theta'), ('g', 'g')]


In [17]:
wanted_module_vars = {"l", "E", "r_p", "r_d", "theta"}

param_map_rows = []

for _, row in param_edges.iterrows():
    vessel = row["vessel"]

    item = {"vessel": vessel}

    for param_name, module_var in row["variable_maps"]:
        if module_var in wanted_module_vars:
            item[module_var + "_param_name"] = param_name

    param_map_rows.append(item)

param_name_df = pd.DataFrame(param_map_rows)

display(param_name_df.head(20))
print(param_name_df.shape)

,vessel,l_param_name,E_param_name,r_p_param_name,r_d_param_name,theta_param_name
0,ascending_aorta_C0,l_ascending_aorta_C0,E_ascending_aorta_C0,rp_ascending_aorta_C0,rd_ascending_aorta_C0,z_ascending_aorta_C0
1,aortic_arch_C2,l_aortic_arch_C2,E_aortic_arch_C2,rp_aortic_arch_C2,rd_aortic_arch_C2,z_aortic_arch_C2
2,brachiocephalic_trunk_C4,l_brachiocephalic_trunk_C4,E_brachiocephalic_trunk_C4,rp_brachiocephalic_trunk_C4,rd_brachiocephalic_trunk_C4,z_brachiocephalic_trunk_C4
3,aortic_arch_C46,l_aortic_arch_C46,E_aortic_arch_C46,rp_aortic_arch_C46,rd_aortic_arch_C46,z_aortic_arch_C46
4,aortic_arch_C64,l_aortic_arch_C64,E_aortic_arch_C64,rp_aortic_arch_C64,rd_aortic_arch_C64,z_aortic_arch_C64
5,aortic_arch_C94,l_aortic_arch_C94,E_aortic_arch_C94,rp_aortic_arch_C94,rd_aortic_arch_C94,z_aortic_arch_C94
6,thoracic_aorta_C96,l_thoracic_aorta_C96,E_thoracic_aorta_C96,rp_thoracic_aorta_C96,rd_thoracic_aorta_C96,z_thoracic_aorta_C96
7,thoracic_aorta_C100,l_thoracic_aorta_C100,E_thoracic_aorta_C100,rp_thoracic_aorta_C100,rd_thoracic_aorta_C100,z_thoracic_aorta_C100
8,thoracic_aorta_C104,l_thoracic_aorta_C104,E_thoracic_aorta_C104,rp_thoracic_aorta_C104,rd_thoracic_aorta_C104,z_thoracic_aorta_C104
9,thoracic_aorta_C108,l_thoracic_aorta_C108,E_thoracic_aorta_C108,rp_thoracic_aorta_C108,rd_thoracic_aorta_C108,z_thoracic_aorta_C108


(103, 6)


In [18]:
params_path = Path("Parameters86.cellml")

params_tree = ET.parse(params_path)
params_root = params_tree.getroot()

ns = {"cellml": "http://www.cellml.org/cellml/1.1#"}

param_components = []

for comp in params_root.findall("cellml:component", ns):

    name = comp.attrib.get("name")

    vars_ = []

    for var in comp.findall("cellml:variable", ns):

        vars_.append({
            "name": var.attrib.get("name"),
            "units": var.attrib.get("units"),
            "initial_value": var.attrib.get("initial_value")
        })

    param_components.append({
        "component": name,
        "n_vars": len(vars_),
        "variables": vars_
    })

params_comp_df = pd.DataFrame(param_components)

print("Components:", len(params_comp_df))

display(params_comp_df[["component", "n_vars"]])

Components: 7


,component,n_vars
0,Parameters_Pulmonary,25
1,Parameters_Heart,48
2,Parameters_Coronary,152
3,Parameters_Systemic,1372
4,Parameters_Venous,385
5,Parameters_Test,25
6,Parameters_Systemic_Baby,63


In [19]:
params_comp_df.iloc[0]["variables"][:20]

[{'name': 'C_pas', 'units': 'm6_per_J', 'initial_value': '0.00135e-6'},
 {'name': 'C_pat', 'units': 'm6_per_J', 'initial_value': '0.0285e-6'},
 {'name': 'C_par', 'units': 'm6_per_J', 'initial_value': '0.0232e-6'},
 {'name': 'C_pcp', 'units': 'm6_per_J', 'initial_value': '0.0684e-6'},
 {'name': 'C_pvn', 'units': 'm6_per_J', 'initial_value': '0.15376e-6'},
 {'name': 'C_pvc', 'units': 'm6_per_J', 'initial_value': '0.01125e-6'},
 {'name': 'R_pas', 'units': 'Js_per_m6', 'initial_value': '0.26664e+6'},
 {'name': 'R_pat', 'units': 'Js_per_m6', 'initial_value': '1.3332e+6'},
 {'name': 'R_par', 'units': 'Js_per_m6', 'initial_value': '6.666e+6'},
 {'name': 'R_pcp', 'units': 'Js_per_m6', 'initial_value': '33.33e+6'},
 {'name': 'R_pvn', 'units': 'Js_per_m6', 'initial_value': '0.8e+6'},
 {'name': 'I_pas', 'units': 'Js2_per_m6', 'initial_value': '0.00693e+6'},
 {'name': 'I_pat', 'units': 'Js2_per_m6', 'initial_value': '0.226644e+6'},
 {'name': 'I_par', 'units': 'Js2_per_m6', 'initial_value': '1e-12'

In [20]:
systemic_vars = []

for comp in params_root.findall("cellml:component", ns):
    if comp.attrib.get("name") == "Parameters_Systemic":
        for var in comp.findall("cellml:variable", ns):
            systemic_vars.append({
                "param_name": var.attrib.get("name"),
                "units": var.attrib.get("units"),
                "initial_value_raw": var.attrib.get("initial_value"),
            })

systemic_param_df = pd.DataFrame(systemic_vars)

systemic_param_df["value"] = pd.to_numeric(
    systemic_param_df["initial_value_raw"],
    errors="coerce"
)

print(systemic_param_df.shape)
display(systemic_param_df.head(20))

(1372, 4)


,param_name,units,initial_value_raw,value
0,C_svl,m6_per_J,0.0037509e-6,3.750900e-09
1,C_svn,m6_per_J,0.1125281e-6,1.125281e-07
2,C_svc,m6_per_J,0.0375094e-6,3.750940e-08
3,C_ivl,m6_per_J,0.0112528e-6,1.125280e-08
4,C_ivn,m6_per_J,0.5626407e-6,5.626407e-07
5,C_ivc,m6_per_J,0.1125281e-6,1.125281e-07
6,R_svl,Js_per_m6,18.662e+6,1.866200e+07
7,R_svn,Js_per_m6,3.999e+6,3.999000e+06
8,R_svc,Js_per_m6,0.06665e+6,6.665000e+04
9,R_ivl,Js_per_m6,5.332e+6,5.332000e+06


In [21]:
needed_names = set()

for col in [
    "l_param_name",
    "E_param_name",
    "r_p_param_name",
    "r_d_param_name",
    "theta_param_name",
]:
    needed_names.update(param_name_df[col].dropna())

available_names = set(systemic_param_df["param_name"])

missing = sorted(needed_names - available_names)

print("Needed:", len(needed_names))
print("Available systemic params:", len(available_names))
print("Missing:", len(missing))

display(missing[:50])

Needed: 512
Available systemic params: 1372
Missing: 0


[]

In [22]:
param_values = systemic_param_df.set_index("param_name")["value"].to_dict()
param_units = systemic_param_df.set_index("param_name")["units"].to_dict()

vessel_params_df = param_name_df.copy()

for logical_name, col in {
    "length_m": "l_param_name",
    "E_Pa": "E_param_name",
    "r_proximal_m": "r_p_param_name",
    "r_distal_m": "r_d_param_name",
    "theta": "theta_param_name",
}.items():
    vessel_params_df[logical_name] = vessel_params_df[col].map(param_values)
    vessel_params_df[logical_name + "_units"] = vessel_params_df[col].map(param_units)

display(vessel_params_df.head(20))

print(vessel_params_df[[
    "length_m", "E_Pa", "r_proximal_m", "r_distal_m", "theta"
]].isna().sum())

,vessel,l_param_name,E_param_name,r_p_param_name,r_d_param_name,theta_param_name,length_m,length_m_units,E_Pa,E_Pa_units,r_proximal_m,r_proximal_m_units,r_distal_m,r_distal_m_units,theta,theta_units
0,ascending_aorta_C0,l_ascending_aorta_C0,E_ascending_aorta_C0,rp_ascending_aorta_C0,rd_ascending_aorta_C0,z_ascending_aorta_C0,0.060000,metre,400000.0,J_per_m3,0.015950,metre,0.013626,metre,0.0,dimensionless
1,aortic_arch_C2,l_aortic_arch_C2,E_aortic_arch_C2,rp_aortic_arch_C2,rd_aortic_arch_C2,z_aortic_arch_C2,0.014476,metre,400000.0,J_per_m3,0.013626,metre,0.012955,metre,90.0,dimensionless
2,brachiocephalic_trunk_C4,l_brachiocephalic_trunk_C4,E_brachiocephalic_trunk_C4,rp_brachiocephalic_trunk_C4,rd_brachiocephalic_trunk_C4,z_brachiocephalic_trunk_C4,0.047382,metre,400000.0,J_per_m3,0.006728,metre,0.006159,metre,0.0,dimensionless
3,aortic_arch_C46,l_aortic_arch_C46,E_aortic_arch_C46,rp_aortic_arch_C46,rd_aortic_arch_C46,z_aortic_arch_C46,0.009608,metre,400000.0,J_per_m3,0.012955,metre,0.012568,metre,90.0,dimensionless
4,aortic_arch_C64,l_aortic_arch_C64,E_aortic_arch_C64,rp_aortic_arch_C64,rd_aortic_arch_C64,z_aortic_arch_C64,0.006980,metre,400000.0,J_per_m3,0.012568,metre,0.012288,metre,90.0,dimensionless
5,aortic_arch_C94,l_aortic_arch_C94,E_aortic_arch_C94,rp_aortic_arch_C94,rd_aortic_arch_C94,z_aortic_arch_C94,0.043211,metre,400000.0,J_per_m3,0.012288,metre,0.010550,metre,90.0,dimensionless
6,thoracic_aorta_C96,l_thoracic_aorta_C96,E_thoracic_aorta_C96,rp_thoracic_aorta_C96,rd_thoracic_aorta_C96,z_thoracic_aorta_C96,0.009898,metre,400000.0,J_per_m3,0.010550,metre,0.010364,metre,180.0,dimensionless
7,thoracic_aorta_C100,l_thoracic_aorta_C100,E_thoracic_aorta_C100,rp_thoracic_aorta_C100,rd_thoracic_aorta_C100,z_thoracic_aorta_C100,0.007880,metre,400000.0,J_per_m3,0.010364,metre,0.010216,metre,180.0,dimensionless
8,thoracic_aorta_C104,l_thoracic_aorta_C104,E_thoracic_aorta_C104,rp_thoracic_aorta_C104,rd_thoracic_aorta_C104,z_thoracic_aorta_C104,0.015556,metre,400000.0,J_per_m3,0.010216,metre,0.009923,metre,180.0,dimensionless
9,thoracic_aorta_C108,l_thoracic_aorta_C108,E_thoracic_aorta_C108,rp_thoracic_aorta_C108,rd_thoracic_aorta_C108,z_thoracic_aorta_C108,0.005327,metre,400000.0,J_per_m3,0.009923,metre,0.009823,metre,180.0,dimensionless


length_m        0
E_Pa            0
r_proximal_m    0
r_distal_m      0
theta           0
dtype: int64


In [23]:
node_full_df = node_df.merge(
    vessel_params_df[
        [
            "vessel",
            "length_m",
            "E_Pa",
            "r_proximal_m",
            "r_distal_m",
            "theta",
        ]
    ],
    on="vessel",
    how="left"
)

display(node_full_df.head(30))

print(node_full_df.shape)
print(node_full_df[[
    "length_m", "E_Pa", "r_proximal_m", "r_distal_m", "theta"
]].isna().sum())

,vessel,order_index,parent,children,n_children,is_root,is_terminal,length_m,E_Pa,r_proximal_m,r_distal_m,theta
0,ascending_aorta_C0,0,NaN,[aortic_arch_C2],1,True,False,0.060000,400000.0,0.015950,0.013626,0.0
1,aortic_arch_C2,1,ascending_aorta_C0,"[brachiocephalic_trunk_C4, aortic_arch_C46]",2,False,False,0.014476,400000.0,0.013626,0.012955,90.0
2,brachiocephalic_trunk_C4,2,aortic_arch_C2,"[common_carotid_R6, subclavian_R28]",2,False,False,0.047382,400000.0,0.006728,0.006159,0.0
3,aortic_arch_C46,3,aortic_arch_C2,"[aortic_arch_C64, common_carotid_L48]",2,False,False,0.009608,400000.0,0.012955,0.012568,90.0
4,common_carotid_R6,4,brachiocephalic_trunk_C4,"[internal_carotid_R8, external_carotid_T2_R26]",2,False,False,0.081253,200000.0,0.004476,0.003326,0.0
5,subclavian_R28,5,brachiocephalic_trunk_C4,"[subclavian_R30, vertebral_R272]",2,False,False,0.015747,400000.0,0.004896,0.004177,90.0
6,aortic_arch_C64,6,aortic_arch_C46,"[aortic_arch_C94, subclavian_L66]",2,False,False,0.006980,400000.0,0.012568,0.012288,90.0
7,common_carotid_L48,7,aortic_arch_C46,"[internal_carotid_L50, external_carotid_T2_L62]",2,False,False,0.121356,200000.0,0.004476,0.003326,0.0
8,internal_carotid_R8,8,common_carotid_R6,[],0,False,True,0.135108,1600000.0,0.002765,0.001339,0.0
9,external_carotid_T2_R26,9,common_carotid_R6,[],0,False,True,0.061012,800000.0,0.002265,0.002265,0.0


(103, 12)
length_m        0
E_Pa            0
r_proximal_m    0
r_distal_m      0
theta           0
dtype: int64


## Compute derived physical quantities

The raw CellML parameters are converted into quantities that are more useful for a 1D blood-flow solver.

For each vessel,  compute:

- proximal reference area
- distal reference area
- mean reference area
- mean radius
- approximate wall thickness
- pressure-area stiffness coefficient
- local pulse-wave speed
- characteristic impedance
- number of numerical grid points

These derived quantities are not the final simulation yet.  
They prepare the arterial network for the next stage, where pressure and flow will be evolved in time.

In [24]:
node_full_df["A0_proximal_m2"] = np.pi * node_full_df["r_proximal_m"]**2
node_full_df["A0_distal_m2"] = np.pi * node_full_df["r_distal_m"]**2
node_full_df["A0_mean_m2"] = 0.5 * (
    node_full_df["A0_proximal_m2"] + node_full_df["A0_distal_m2"]
)

node_full_df["radius_mean_m"] = 0.5 * (
    node_full_df["r_proximal_m"] + node_full_df["r_distal_m"]
)

display(node_full_df.head(20))

,vessel,order_index,parent,children,n_children,is_root,is_terminal,length_m,E_Pa,r_proximal_m,r_distal_m,theta,A0_proximal_m2,A0_distal_m2,A0_mean_m2,radius_mean_m
0,ascending_aorta_C0,0,NaN,[aortic_arch_C2],1,True,False,0.060000,400000.0,0.015950,0.013626,0.0,0.000799,0.000583,0.000691,0.014788
1,aortic_arch_C2,1,ascending_aorta_C0,"[brachiocephalic_trunk_C4, aortic_arch_C46]",2,False,False,0.014476,400000.0,0.013626,0.012955,90.0,0.000583,0.000527,0.000555,0.013290
2,brachiocephalic_trunk_C4,2,aortic_arch_C2,"[common_carotid_R6, subclavian_R28]",2,False,False,0.047382,400000.0,0.006728,0.006159,0.0,0.000142,0.000119,0.000131,0.006443
3,aortic_arch_C46,3,aortic_arch_C2,"[aortic_arch_C64, common_carotid_L48]",2,False,False,0.009608,400000.0,0.012955,0.012568,90.0,0.000527,0.000496,0.000512,0.012762
4,common_carotid_R6,4,brachiocephalic_trunk_C4,"[internal_carotid_R8, external_carotid_T2_R26]",2,False,False,0.081253,200000.0,0.004476,0.003326,0.0,0.000063,0.000035,0.000049,0.003901
5,subclavian_R28,5,brachiocephalic_trunk_C4,"[subclavian_R30, vertebral_R272]",2,False,False,0.015747,400000.0,0.004896,0.004177,90.0,0.000075,0.000055,0.000065,0.004537
6,aortic_arch_C64,6,aortic_arch_C46,"[aortic_arch_C94, subclavian_L66]",2,False,False,0.006980,400000.0,0.012568,0.012288,90.0,0.000496,0.000474,0.000485,0.012428
7,common_carotid_L48,7,aortic_arch_C46,"[internal_carotid_L50, external_carotid_T2_L62]",2,False,False,0.121356,200000.0,0.004476,0.003326,0.0,0.000063,0.000035,0.000049,0.003901
8,internal_carotid_R8,8,common_carotid_R6,[],0,False,True,0.135108,1600000.0,0.002765,0.001339,0.0,0.000024,0.000006,0.000015,0.002052
9,external_carotid_T2_R26,9,common_carotid_R6,[],0,False,True,0.061012,800000.0,0.002265,0.002265,0.0,0.000016,0.000016,0.000016,0.002265


In [25]:
print("Length range:", node_full_df["length_m"].min(), node_full_df["length_m"].max())
print("Radius range:", node_full_df["radius_mean_m"].min(), node_full_df["radius_mean_m"].max())
print("A0 range:", node_full_df["A0_mean_m2"].min(), node_full_df["A0_mean_m2"].max())
print("E range:", node_full_df["E_Pa"].min(), node_full_df["E_Pa"].max())

Length range: 0.00279812 0.386389
Radius range: 0.000558491 0.0147878
A0 range: 9.799010669147214e-07 0.0006912438074667773
E range: 200000.0 1600000.0


## Save exploratory parsed tables

Before building the final parser function,  save the parsed vessel table and topology table as CSV files.

These files are useful for inspection, debugging, and quick validation.

The final solver will use a more compact pickle file, but CSV exports make it easier to manually check whether vessel names, parameters, and connections look reasonable.

In [26]:
node_full_df.to_csv("adan86_parsed_from_cellml.csv", index=False)

print("Saved:", "adan86_parsed_from_cellml.csv")
print("Rows:", len(node_full_df))
print("Columns:", list(node_full_df.columns))

Saved: adan86_parsed_from_cellml.csv
Rows: 103
Columns: ['vessel', 'order_index', 'parent', 'children', 'n_children', 'is_root', 'is_terminal', 'length_m', 'E_Pa', 'r_proximal_m', 'r_distal_m', 'theta', 'A0_proximal_m2', 'A0_distal_m2', 'A0_mean_m2', 'radius_mean_m']


In [27]:
topology_df[["parent", "child"]].to_csv("adan86_edges_from_cellml.csv", index=False)

print("Saved:", "adan86_edges_from_cellml.csv")
print("Edges:", len(topology_df))

Saved: adan86_edges_from_cellml.csv
Edges: 102


## Build the complete reusable parser

The exploratory cells above helped identify how the CellML model stores vessels, topology, and parameters.

Now  combine the full parsing workflow into a single reusable function.

This function takes:

```python
main_ADAN-86.cellml
Parameters86.cellml

In [ ]:
def parse_adan86_cellml(main_path, params_path):
    """
    Parse ADAN-86 CellML files into vessel topology and geometry tables.

    Loads the main CellML model and parameter file, extracts vessel
    components, parent-child topology, systemic vessel parameters,
    derived geometry, and graph structures.

    Args:
        main_path: Path to the main ADAN-86 CellML file.
        params_path: Path to the Parameters86 CellML file.

    Returns:
        (node_full_df, topology_df, children_map, order)
        Vessel table, topology edge table, children map, and traversal order.
    """
    # LOAD MAIN CELLML
    ns = {"cellml": "http://www.cellml.org/cellml/1.1#"}
    tree = ET.parse(main_path)
    root = tree.getroot()

    # PARSE COMPONENTS
    components = []

    for comp in root.findall("cellml:component", ns):
        name = comp.attrib.get("name")

        variables = [var.attrib.get("name")for var in comp.findall("cellml:variable", ns)]

        components.append({
            "component": name,
            "variables": variables,
            "n_variables": len(variables)
        })
    comp_df = pd.DataFrame(components)

    # FILTER VESSEL COMPONENTS
    required_vars = {"u", "v", "E", "r_p", "r_d", "h", "l", "q_0"}

    vessel_df = comp_df[comp_df["variables"].apply(lambda xs: required_vars.issubset(set(xs)))].copy()

    # PARSE CONNECTIONS
    connections = []
    for conn in root.findall("cellml:connection", ns):
        comps = conn.findall("cellml:map_components", ns)

        if len(comps) == 0:
            continue
            
        comp1 = comps[0].attrib.get("component_1")
        comp2 = comps[0].attrib.get("component_2")
        var_maps = []

        for vm in conn.findall("cellml:map_variables", ns):
            v1 = vm.attrib.get("variable_1")
            v2 = vm.attrib.get("variable_2")
            var_maps.append((v1, v2))

        connections.append({
            "component_1": comp1,
            "component_2": comp2,
            "variable_maps": var_maps
        })
    conn_df = pd.DataFrame(connections)

    # TOPOLOGY EDGES

    def clean_module_name(name):
        return name.replace("_module", "")

    def is_module_edge(row):
        c1 = row["component_1"]
        c2 = row["component_2"]

        if not isinstance(c1, str) or not isinstance(c2, str):
            return False

        if not c1.endswith("_module"):
            return False

        if not c2.endswith("_module"):
            return False

        maps = set(row["variable_maps"])

        valid_patterns = [
            ("v_out", "v"),
            ("v_out_1", "v"),
            ("v_out_2", "v"),
            ("u", "u_in"),
        ]

        return any(p in maps for p in valid_patterns)

    topo_edges = conn_df[conn_df.apply(is_module_edge, axis=1)].copy()

    topo_edges["parent"] = topo_edges["component_1"].apply(clean_module_name)
    topo_edges["child"] = topo_edges["component_2"].apply(clean_module_name)

    topology_df = topo_edges[["parent", "child", "variable_maps"]].copy()

    # GRAPH STRUCTURES
    all_vessels = set(vessel_df["component"])
    children_map = defaultdict(list)

    for _, row in topology_df.iterrows():
        children_map[row["parent"]].append(row["child"])

    children_map = dict(children_map)
    children = set(topology_df["child"])
    roots = sorted(all_vessels - children)
    root_vessel = roots[0]
    order = []
    queue = deque([root_vessel])

    while queue:
        node = queue.popleft()
        order.append(node)
        for child in children_map.get(node, []):
            queue.append(child)

    parent_map = {}
    for _, row in topology_df.iterrows():
        parent_map[row["child"]] = row["parent"]

    node_df = pd.DataFrame({"vessel": order})

    node_df["order_index"] = range(len(node_df))
    node_df["parent"] = node_df["vessel"].map(parent_map)
    node_df["children"] = node_df["vessel"].map(lambda x: children_map.get(x, []))

    node_df["n_children"] = node_df["children"].apply(len)
    node_df["is_root"] = node_df["parent"].isna()
    node_df["is_terminal"] = node_df["n_children"].eq(0)

    # LOAD PARAMETER FILE
    params_tree = ET.parse(params_path)
    params_root = params_tree.getroot()

    systemic_vars = []

    for comp in params_root.findall("cellml:component", ns):

        if comp.attrib.get("name") != "Parameters_Systemic":
            continue

        for var in comp.findall("cellml:variable", ns):

            systemic_vars.append({
                "param_name": var.attrib.get("name"),
                "units": var.attrib.get("units"),
                "value": pd.to_numeric(
                    var.attrib.get("initial_value"),
                    errors="coerce"
                )
            })

    systemic_param_df = pd.DataFrame(systemic_vars)

    param_values = systemic_param_df.set_index("param_name")["value"].to_dict()

    # PARAMETER MAPPINGS

    param_edges = conn_df[(conn_df["component_1"] == "Parameters_Systemic")
        &(conn_df["component_2"].str.endswith("_module", na=False))].copy()

    param_edges["vessel"] = param_edges["component_2"].str.replace("_module", "", regex=False)

    wanted_module_vars = {"l", "E", "r_p", "r_d", "theta"}

    param_rows = []

    for _, row in param_edges.iterrows():

        item = {"vessel": row["vessel"]}

        for param_name, module_var in row["variable_maps"]:

            if module_var in wanted_module_vars:

                item[module_var] = param_values.get(param_name)

        param_rows.append(item)

    vessel_params_df = pd.DataFrame(param_rows)

    vessel_params_df = vessel_params_df.rename(columns={
        "l": "length_m",
        "E": "E_Pa",
        "r_p": "r_proximal_m",
        "r_d": "r_distal_m",
        "theta": "theta",
    })
    # FINAL MERGE
    node_full_df = node_df.merge(
        vessel_params_df,
        on="vessel",
        how="left"
    )

    # DERIVED GEOMETRY


    node_full_df["A0_proximal_m2"] = (np.pi * node_full_df["r_proximal_m"]**2)

    node_full_df["A0_distal_m2"] = (np.pi * node_full_df["r_distal_m"]**2)

    node_full_df["A0_mean_m2"] = 0.5 * (node_full_df["A0_proximal_m2"]+ node_full_df["A0_distal_m2"])

    node_full_df["radius_mean_m"] = 0.5 * (node_full_df["r_proximal_m"] + node_full_df["r_distal_m"])

    # ASSERTS

    assert len(node_full_df) == 103
    assert len(topology_df) == 102
    assert node_full_df["is_root"].sum() == 1
    assert node_full_df["is_terminal"].sum() == 43
    assert node_full_df[["length_m", "E_Pa", "r_proximal_m", "r_distal_m"]].isna().sum().sum() == 0

    return (node_full_df, topology_df, children_map, order)

In [29]:
# PARSE ADAN-86 CELLML


node_full_df, topology_df, children_map, order = parse_adan86_cellml(
    "main_ADAN-86.cellml",
    "Parameters86.cellml"
)

print("node_full_df:", node_full_df.shape)
print("topology_df:", topology_df.shape)
print("children_map:", len(children_map))
print("order:", len(order))

display(node_full_df.head())
display(topology_df.head())

node_full_df: (103, 16)
topology_df: (102, 3)
children_map: 60
order: 103


,vessel,order_index,parent,children,n_children,is_root,is_terminal,length_m,E_Pa,r_proximal_m,r_distal_m,theta,A0_proximal_m2,A0_distal_m2,A0_mean_m2,radius_mean_m
0,ascending_aorta_C0,0,NaN,[aortic_arch_C2],1,True,False,0.060000,400000.0,0.015950,0.013626,0.0,0.000799,0.000583,0.000691,0.014788
1,aortic_arch_C2,1,ascending_aorta_C0,"[brachiocephalic_trunk_C4, aortic_arch_C46]",2,False,False,0.014476,400000.0,0.013626,0.012955,90.0,0.000583,0.000527,0.000555,0.013290
2,brachiocephalic_trunk_C4,2,aortic_arch_C2,"[common_carotid_R6, subclavian_R28]",2,False,False,0.047382,400000.0,0.006728,0.006159,0.0,0.000142,0.000119,0.000131,0.006443
3,aortic_arch_C46,3,aortic_arch_C2,"[aortic_arch_C64, common_carotid_L48]",2,False,False,0.009608,400000.0,0.012955,0.012568,90.0,0.000527,0.000496,0.000512,0.012762
4,common_carotid_R6,4,brachiocephalic_trunk_C4,"[internal_carotid_R8, external_carotid_T2_R26]",2,False,False,0.081253,200000.0,0.004476,0.003326,0.0,0.000063,0.000035,0.000049,0.003901


,parent,child,variable_maps
51,aortic_arch_C2,brachiocephalic_trunk_C4,"[(v_out_1, v), (u, u_in)]"
52,aortic_arch_C2,aortic_arch_C46,"[(v_out_2, v), (u, u_in)]"
53,brachiocephalic_trunk_C4,common_carotid_R6,"[(v_out_1, v), (u, u_in)]"
54,brachiocephalic_trunk_C4,subclavian_R28,"[(v_out_2, v), (u, u_in)]"
55,aortic_arch_C46,aortic_arch_C64,"[(v_out_1, v), (u, u_in)]"


## Build solver-ready arrays

The 1D solver should work with numerical arrays rather than DataFrame columns.

Convert the parsed vessel table into ordered NumPy arrays:

- reference area
- stiffness
- vessel length
- grid spacing
- wave speed
- terminal vessel indices
- root vessel index

The order of these arrays matches the arterial traversal order.  
This makes it easier to update each vessel during the time-stepping simulation.

In [30]:
# BUILD SOLVER-READY ARRAYS

# Index maps

vessel_to_idx = {vessel: i for i, vessel in enumerate(order)}

node_full_df["idx"] = node_full_df["vessel"].map(vessel_to_idx)

node_full_df["parent_idx"] = node_full_df["parent"].map(vessel_to_idx)
node_full_df["parent_idx"] = node_full_df["parent_idx"].fillna(-1).astype(int)

node_full_df["children_idx"] = node_full_df["children"].apply(lambda xs: [vessel_to_idx[x] for x in xs])

children_map_idx = {vessel_to_idx[parent]: [vessel_to_idx[ch] for ch in children] for parent, children in children_map.items()}

parent_idx = node_full_df["parent_idx"].to_numpy(dtype=int)
children_idx = node_full_df["children_idx"].to_list()

terminal_idx = np.array([i for i, children in enumerate(children_idx) if len(children) == 0], dtype=int)

root_idx = np.array([i for i, p in enumerate(parent_idx) if p == -1], dtype=int)

# Physical arrays

A0v = node_full_df["A0_mean_m2"].to_numpy(dtype=float)
Lv = node_full_df["length_m"].to_numpy(dtype=float)
rv = node_full_df["radius_mean_m"].to_numpy(dtype=float)
Ev = node_full_df["E_Pa"].to_numpy(dtype=float)

# Blood / wall constants

RHO = 1060.0 # kg/m^3
MMHG = 133.322 # Pa / mmHg
H_OVER_R = 0.10 # wall thickness approx = 10% radius

hv = H_OVER_R * rv

# Pressure-area law:
# P = beta * (sqrt(A) - sqrt(A0))
betav = (4.0 / 3.0) * np.sqrt(np.pi) * Ev * hv / A0v

# Moens-Korteweg / local wave speed at A0
c0v = np.sqrt(Ev * hv / (2.0 * RHO * rv))

# Characteristic impedance
Z0v = RHO * c0v / A0v

# Numerical grid

DX_TARGET = 0.005  # 5 mm target spacing
Nv = np.maximum(3, np.ceil(Lv / DX_TARGET).astype(int) + 1)
dxv = Lv / (Nv - 1)

# CFL time step
CFL_TARGET = 0.4
min_dx_over_c0 = np.min(dxv / c0v)
dt_global = CFL_TARGET * min_dx_over_c0
cfl_actual = c0v * dt_global / dxv

# Diagnostics


print("A0v shape:", A0v.shape)
print("Lv shape:", Lv.shape)
print("rv shape:", rv.shape)
print("Ev shape:", Ev.shape)
print("betav shape:", betav.shape)
print("c0v shape:", c0v.shape)
print("Z0v shape:", Z0v.shape)
print("Nv shape:", Nv.shape)
print("dxv shape:", dxv.shape)

print("\nA0 first:", A0v[:5])
print("L first:", Lv[:5])

print("\nRoot idx:", root_idx)
print("Root vessel:", [order[i] for i in root_idx])
print("Terminals:", len(terminal_idx))
print("First terminals:", [(i, order[i]) for i in terminal_idx[:10]])

print("\nc0 range:", c0v.min(), c0v.max())
print("Z0 range:", Z0v.min(), Z0v.max())
print("Nv range:", Nv.min(), Nv.max())
print("dx range:", dxv.min(), dxv.max())

print("\ndt_global:", dt_global)
print("min dx/c0:", min_dx_over_c0)
print("max CFL actual:", cfl_actual.max())

A0v shape: (103,)
Lv shape: (103,)
rv shape: (103,)
Ev shape: (103,)
betav shape: (103,)
c0v shape: (103,)
Z0v shape: (103,)
Nv shape: (103,)
dxv shape: (103,)

A0 first: [6.91243807e-04 5.55255153e-04 1.30685972e-04 5.11760144e-04
 4.88435637e-05]
L first: [0.06       0.0144757  0.0473822  0.00960849 0.0812532 ]

Root idx: [0]
Root vessel: ['ascending_aorta_C0']
Terminals: 43
First terminals: [(np.int64(8), 'internal_carotid_R8'), (np.int64(9), 'external_carotid_T2_R26'), (np.int64(11), 'vertebral_R272'), (np.int64(14), 'internal_carotid_L50'), (np.int64(15), 'external_carotid_T2_L62'), (np.int64(19), 'vertebral_L2'), (np.int64(22), 'posterior_intercostal_T1_R98'), (np.int64(25), 'radial_T1_R44'), (np.int64(27), 'posterior_intercostal_T1_L102'), (np.int64(30), 'ulnar_T2_R42')]

c0 range: 3.071475584169756 8.687444855261388
Z0 range: 6660957.716441938 6414531479.234209
Nv range: 3 79
dx range: 0.00139906 0.005

dt_global: 0.00012883511995154115
min dx/c0: 0.00032208779987885286
max CFL

## Sanity checks

Before saving the parsed network,  run basic validation checks.

These checks confirm that:

- all expected vessel segments are present
- topology has the expected number of edges
- terminal vessels are identified
- the root vessel is unique
- all solver arrays have consistent length
- physical quantities are positive and finite
- the global time step satisfies the CFL-based stability constraint

These checks are important because errors in topology or vessel parameters would later cause unstable or unrealistic wave propagation.

In [31]:
# PARSED DATA SANITY CHECKS

assert len(order) == 103
assert len(topology_df) == 102
assert len(terminal_idx) == 43
assert len(root_idx) == 1

assert A0v.shape[0] == len(order)
assert Lv.shape[0] == len(order)
assert rv.shape[0] == len(order)
assert Ev.shape[0] == len(order)
assert betav.shape[0] == len(order)
assert c0v.shape[0] == len(order)
assert Z0v.shape[0] == len(order)
assert Nv.shape[0] == len(order)
assert dxv.shape[0] == len(order)

for name, arr in [
    ("A0v", A0v),
    ("Lv", Lv),
    ("rv", rv),
    ("Ev", Ev),
    ("betav", betav),
    ("c0v", c0v),
    ("Z0v", Z0v),
    ("dxv", dxv),
]:
    if np.isnan(arr).any():
        raise ValueError(f"{name} contains NaN")
    if np.any(arr <= 0):
        raise ValueError(f"{name} contains non-positive values")

children_count = np.array([len(x) for x in children_idx])

print("Children count distribution:")
for n in sorted(set(children_count)):
    print(f"{n} children:", np.sum(children_count == n))

print("\n✓ ADAN parsed data looks valid")

Children count distribution:
0 children: 43
1 children: 18
2 children: 42

✓ ADAN parsed data looks valid


## Save parsed ADAN data for the solver

The final output of this notebook is saved as:

```python
adan_parsed.pkl

In [32]:
# SAVE PARSED ADAN DATA

OUTPUT_PATH = Path("adan_parsed.pkl")

adan_data = {
    # DataFrames
    "node_full_df": node_full_df.copy(),
    "topology_df": topology_df.copy(),

    # Order / indexing
    "order": order.copy(),
    "vessel_to_idx": vessel_to_idx.copy(),

    # Physical arrays
    "A0v": A0v.copy(),
    "Lv": Lv.copy(),
    "rv": rv.copy(),
    "Ev": Ev.copy(),
    "betav": betav.copy(),
    "c0v": c0v.copy(),
    "Z0v": Z0v.copy(),

    # Numerical grid
    "dxv": dxv.copy(),
    "Nv": Nv.copy(),
    "dt_global": dt_global,

    # Graph/index arrays
    "parent_idx": parent_idx.copy(),
    "children_idx": [list(x) for x in children_idx],
    "children_map_idx": {
        int(k): [int(c) for c in v]
        for k, v in children_map_idx.items()
    },
    "terminal_idx": terminal_idx.copy(),
    "root_idx": root_idx.copy(),

    # Constants
    "RHO": RHO,
    "MMHG": MMHG,
    "H_OVER_R": H_OVER_R,
    "DX_TARGET": DX_TARGET,
    "CFL_TARGET": CFL_TARGET,
}

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(adan_data, f)

print("Saved:", OUTPUT_PATH.resolve())
print("File size MB:", OUTPUT_PATH.stat().st_size / 1024 / 1024)

Saved: D:\code\adan_project\adan_parsed.pkl
File size MB: 0.03179645538330078


In [33]:
# RELOAD TEST

with open("adan_parsed.pkl", "rb") as f:
    test_data = pickle.load(f)

print("Loaded keys:")
print(test_data.keys())

print("\nLoaded vessels:", len(test_data["order"]))
print("Loaded topology rows:", len(test_data["topology_df"]))
print("Loaded terminals:", len(test_data["terminal_idx"]))

assert len(test_data["order"]) == 103
assert len(test_data["topology_df"]) == 102
assert len(test_data["terminal_idx"]) == 43
assert test_data["A0v"].shape == A0v.shape
assert test_data["Nv"].shape == Nv.shape

print("\n✓ Reload test passed")

Loaded keys:
dict_keys(['node_full_df', 'topology_df', 'order', 'vessel_to_idx', 'A0v', 'Lv', 'rv', 'Ev', 'betav', 'c0v', 'Z0v', 'dxv', 'Nv', 'dt_global', 'parent_idx', 'children_idx', 'children_map_idx', 'terminal_idx', 'root_idx', 'RHO', 'MMHG', 'H_OVER_R', 'DX_TARGET', 'CFL_TARGET'])

Loaded vessels: 103
Loaded topology rows: 102
Loaded terminals: 43

✓ Reload test passed
